In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive as google_drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    google_drive.mount("/content/drive", force_remount=False)
    WORKSPACE = Path("/content/drive/MyDrive/Zhong et al. 2025 - Neuromatch Team Workspace")
    DATASET_ROOT = None
    CACHE = None
else:
    WORKSPACE = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "code").is_dir())
    mounted = WORKSPACE / "data" / "drive"
    DATASET_ROOT = str(mounted) if mounted.is_dir() else None
    CACHE = str(WORKSPACE / "data" / "cache")

if str(WORKSPACE / "code") not in sys.path:
    sys.path.insert(0, str(WORKSPACE / "code"))

import drive
from joiner import Joiner
from dprime import (
    eligible_trial_frames,
    ordinal_trial_pairs,
    paired_trial_windows,
    windowed_svd_dprime,
)

db = drive.setup(root=DATASET_ROOT, cache=CACHE, mount=IN_COLAB)
db

In [ ]:
sessions = db.query("""
    SELECT b.behavior_session_id, b.recording_id, b.experiment, b.behavior_key,
           b.cohort, b.mouse, e.stage, e.moment
    FROM behavior_sessions AS b
    JOIN recordings AS r USING (recording_id)
    LEFT JOIN experiments AS e USING (experiment)
    WHERE r.has_reduced_neural AND r.has_behavior AND r.has_retinotopy
    ORDER BY b.cohort, b.mouse, e.stage, e.moment, b.experiment
""")
sessions

In [ ]:
experiment_mice = db.query("""
    SELECT
        b.cohort,
        b.experiment,
        e.stage,
        e.moment,
        COUNT(DISTINCT b.mouse) AS mouse_count,
        LIST(DISTINCT b.mouse ORDER BY b.mouse) AS mice,
        COUNT(DISTINCT b.behavior_session_id) AS sessions,
        COUNT(DISTINCT b.recording_id) AS recordings
    FROM behavior_sessions AS b
    JOIN recordings AS r USING (recording_id)
    LEFT JOIN experiments AS e USING (experiment)
    WHERE r.has_reduced_neural
      AND r.has_behavior
      AND r.has_retinotopy
    GROUP BY ALL
    ORDER BY b.cohort, e.stage, e.moment, b.experiment
""")
experiment_mice

In [ ]:
db.query("""
    SELECT filename,
           COUNT(*) AS sessions,
           COUNT(DISTINCT recording_id) AS recordings,
           COUNT(DISTINCT mouse) AS mice
    FROM behavior_sessions
    GROUP BY filename
    ORDER BY sessions DESC, filename
""")

In [ ]:
def leaf_circle_pairs(session):
    joiner = Joiner(
        db,
        session.recording_id,
        experiment=session.experiment,
        behavior_key=session.behavior_key,
    )
    pairs = ordinal_trial_pairs(eligible_trial_frames(joiner.frames))
    return pairs.assign(behavior_session_id=session.behavior_session_id), len(joiner.neurons)


paired = []
neurons = []
for session in sessions.itertuples(index=False):
    try:
        session_pairs, n_neurons = leaf_circle_pairs(session)
    except Exception as error:
        print(f"skip {session.behavior_session_id}: {type(error).__name__}: {error}")
        continue
    paired.append(session_pairs)
    neurons.append({"behavior_session_id": session.behavior_session_id, "n_neurons": n_neurons})

db.register("session_pairs", pd.concat(paired, ignore_index=True))
db.register("session_neurons", pd.DataFrame(neurons))
pairs = db.query("""
    SELECT b.behavior_session_id, b.recording_id, b.experiment, b.behavior_key,
           b.cohort, b.mouse, e.stage, e.moment, b.date,
           p.pair_id, p.trial_a_id, p.trial_b_id,
           p.first_trial_id, p.last_trial_id, p.trial_midpoint,
           p.role_a, p.role_b
    FROM session_pairs AS p
    JOIN behavior_sessions AS b USING (behavior_session_id)
    LEFT JOIN experiments AS e USING (experiment)
    ORDER BY b.cohort, b.mouse, b.experiment, p.pair_id
""")
db.register("pairs", pairs)
pairs

In [ ]:
db.query("""
    SELECT COUNT(DISTINCT behavior_session_id) AS sessions,
           COUNT(DISTINCT recording_id) AS recordings,
           COUNT(DISTINCT mouse) AS mice,
           COUNT(*) AS leaf_circle_pairs
    FROM pairs
""")

In [ ]:
db.query("""
    SELECT cohort, mouse,
           COUNT(DISTINCT behavior_session_id) AS sessions,
           COUNT(*) AS pairs
    FROM pairs
    GROUP BY cohort, mouse
    ORDER BY cohort, mouse
""")

In [ ]:
for moment in ("before", "after"):
    print(f"{moment} learning")
    print(
        db.query(
            "SELECT * FROM pairs WHERE experiment LIKE ? "
            "ORDER BY cohort, mouse, experiment, pair_id",
            [f"%{moment}_learning"],
        ),
        end="\n\n",
    )

In [ ]:
trial_counts = db.query("""
    SELECT mouse,
           COUNT(*) FILTER (WHERE experiment LIKE '%before_learning') AS before,
           COUNT(*) FILTER (WHERE experiment LIKE '%after_learning')  AS after
    FROM pairs
    WHERE experiment LIKE '%_learning'
    GROUP BY mouse
    ORDER BY mouse
""")
print(trial_counts.to_string(index=False))

x = np.arange(len(trial_counts))
width = 0.4
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(x - width / 2, trial_counts["before"], width, label="before learning", color="#6B6B6B")
ax.bar(x + width / 2, trial_counts["after"], width, label="after learning", color="#2C6FBB")
ax.set_xticks(x)
ax.set_xticklabels(trial_counts["mouse"], rotation=45, ha="right")
ax.set_ylabel("leaf-circle pairs")
ax.set_title("Trial count by mouse: before vs after learning")
ax.legend()
ax.grid(axis="y", alpha=0.2)
plt.show()

In [ ]:
neuron_counts = db.query("""
    SELECT b.mouse,
           ROUND(AVG(n.n_neurons) FILTER (WHERE b.experiment LIKE '%before_learning')) AS before,
           ROUND(AVG(n.n_neurons) FILTER (WHERE b.experiment LIKE '%after_learning'))  AS after
    FROM session_neurons AS n
    JOIN behavior_sessions AS b USING (behavior_session_id)
    WHERE b.experiment LIKE '%_learning'
    GROUP BY b.mouse
    ORDER BY b.mouse
""")
print(neuron_counts.to_string(index=False))

x = np.arange(len(neuron_counts))
width = 0.4
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(x - width / 2, neuron_counts["before"], width, label="before learning", color="#6B6B6B")
ax.bar(x + width / 2, neuron_counts["after"], width, label="after learning", color="#2C6FBB")
ax.set_xticks(x)
ax.set_xticklabels(neuron_counts["mouse"], rotation=45, ha="right")
ax.set_ylabel("mean neurons / session")
ax.set_title("Neuron count by mouse: before vs after learning")
ax.legend()
ax.grid(axis="y", alpha=0.2)
plt.show()

In [ ]:
selection = db.query("""
    WITH session_pairs AS (
        SELECT p.cohort, p.mouse, e.stage, p.moment,
               p.recording_id, p.experiment, p.behavior_key, p.behavior_session_id, p.date,
               COUNT(*) AS n_pairs
        FROM pairs AS p
        JOIN experiments AS e USING (experiment)
        WHERE p.cohort IN ('supervised', 'unsupervised', 'grating')
          AND p.moment IN ('before', 'after')
          AND e.stage IN ('train1', 'train2')
        GROUP BY ALL
    ),
    matched_mouse AS (
        SELECT bef.cohort, bef.mouse, bef.stage,
               bef.recording_id AS before_recording, bef.experiment AS before_experiment,
               bef.behavior_key AS before_key, bef.behavior_session_id AS before_session,
               bef.date AS before_date, bef.n_pairs AS before_pairs,
               aft.recording_id AS after_recording, aft.experiment AS after_experiment,
               aft.behavior_key AS after_key, aft.behavior_session_id AS after_session,
               aft.date AS after_date, aft.n_pairs AS after_pairs
        FROM session_pairs AS bef
        JOIN session_pairs AS aft
          ON bef.cohort = aft.cohort AND bef.mouse = aft.mouse AND bef.stage = aft.stage
         AND bef.moment = 'before' AND aft.moment = 'after' AND bef.date < aft.date
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY bef.cohort, bef.mouse
            ORDER BY CASE bef.stage WHEN 'train1' THEN 0 WHEN 'train2' THEN 1 ELSE 2 END,
                     LEAST(bef.n_pairs, aft.n_pairs) DESC, bef.date DESC, aft.date ASC
        ) = 1
    )
    SELECT cohort, mouse, stage, 'before' AS moment,
           before_recording AS recording_id, before_experiment AS experiment,
           before_key AS behavior_key, before_date AS date, before_pairs AS n_pairs
    FROM matched_mouse
    UNION ALL
    SELECT cohort, mouse, stage, 'after' AS moment,
           after_recording, after_experiment, after_key, after_date, after_pairs
    FROM matched_mouse
    ORDER BY cohort, mouse, moment
""")

MICE_BY_COHORT = selection.groupby("cohort").mouse.nunique().to_dict()
assert selection.groupby(["cohort", "mouse"]).moment.nunique().eq(2).all()
selection

In [ ]:
AREAS = ("V1", "mHV", "lHV", "aHV")
NEURON_CAP = 8000
THRESHOLD = 0.3
WINDOW_SIZES = (5, 10, 20)


def session_dynamics(session):
    joiner = Joiner(
        db,
        session.recording_id,
        experiment=session.experiment,
        behavior_key=session.behavior_key,
    )
    frames = eligible_trial_frames(joiner.frames)
    pairs = ordinal_trial_pairs(frames)
    visual = joiner.neurons[joiner.neurons.area_group.isin(AREAS)]
    if len(visual) > NEURON_CAP:
        visual = visual.sample(n=NEURON_CAP, random_state=0)
    neuron_ids = visual.neuron_id.to_numpy()
    dynamics = {}
    for size in WINDOW_SIZES:
        windows = paired_trial_windows(pairs, size=size, stride=1)
        dprime = windowed_svd_dprime(joiner.U, joiner.V, frames, pairs, windows, neuron_ids=neuron_ids)
        dynamics[size] = (windows["progress"].to_numpy(), dprime)
    return dynamics


records = []
distributions = {}
for session in selection.itertuples(index=False):
    for size, (progress, dprime) in session_dynamics(session).items():
        for window_index, fraction in enumerate(progress):
            values = dprime[window_index]
            records.append({
                "cohort": session.cohort,
                "moment": session.moment,
                "mouse": session.mouse,
                "window_size": size,
                "progress": float(fraction),
                "mean_dprime": float(values.mean()),
                "mean_abs_dprime": float(np.abs(values).mean()),
                "frac_selective": float((np.abs(values) >= THRESHOLD).mean()),
            })
        if size == 10:
            distributions[(session.cohort, session.mouse, session.moment)] = (progress, dprime)

history = pd.DataFrame(records)
history

In [ ]:
COHORTS = ["supervised", "unsupervised", "grating"]
MOMENT_COLOR = {"before": "#6B6B6B", "after": "#2C6FBB"}
PROGRESS_GRID = np.linspace(0.0, 1.0, 101)


def mouse_curves(frame, column):
    return np.vstack([
        np.interp(PROGRESS_GRID, track.progress, track[column])
        for _, track in frame.sort_values("progress").groupby("mouse")
    ])


def plot_mouse_average(ax, frame, column, *, color, label=None, linestyle="-"):
    curves = mouse_curves(frame, column)
    mean = curves.mean(axis=0)
    sem = curves.std(axis=0, ddof=1) / np.sqrt(len(curves))
    for curve in curves:
        ax.plot(PROGRESS_GRID, curve, color=color, ls=linestyle, lw=0.7, alpha=0.18)
    ax.plot(PROGRESS_GRID, mean, color=color, ls=linestyle, lw=1.8, label=label)
    ax.fill_between(PROGRESS_GRID, mean - sem, mean + sem, color=color, alpha=0.10)

fig, axes = plt.subplots(len(WINDOW_SIZES), len(COHORTS), figsize=(13, 8), sharex=True, sharey=True)
for row_index, size in enumerate(WINDOW_SIZES):
    for col_index, cohort in enumerate(COHORTS):
        ax = axes[row_index, col_index]
        for moment, color in MOMENT_COLOR.items():
            track = history[(history.window_size == size) & (history.cohort == cohort) & (history.moment == moment)]
            if len(track):
                plot_mouse_average(ax, track, "mean_abs_dprime", color=color, label=moment)
        ax.grid(alpha=0.2)
        if row_index == 0:
            ax.set_title(f"{cohort} (n={MICE_BY_COHORT[cohort]})")
        if col_index == 0:
            ax.set_ylabel(f"window size {size}\nmean |d'| (exploratory)")
        if row_index == len(WINDOW_SIZES) - 1:
            ax.set_xlabel("within-session progress")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Within-session d' dynamics (mean |d'|, exploratory), pooled visual, sizes 5/10/20 sliding - all available matched mice")
plt.show()

In [ ]:
from matplotlib import animation
from IPython.display import HTML

panels = [(cohort, moment) for moment in ("before", "after") for cohort in COHORTS]
checkpoints = np.linspace(0.0, 1.0, 24)
pooled_values = np.concatenate([dprime.ravel() for _, dprime in distributions.values()])
lo, hi = np.percentile(pooled_values, [1, 99])
bins = np.linspace(lo, hi, 46)


def window_at(cohort, moment, fraction):
    values = []
    for (current_cohort, _, current_moment), (progress, dprime) in distributions.items():
        if current_cohort == cohort and current_moment == moment:
            values.append(dprime[int(np.argmin(np.abs(progress - fraction)))])
    return np.concatenate(values)


ymax = max(
    np.histogram(window_at(*key, f), bins=bins, range=(lo, hi))[0].max()
    for key in panels
    for f in checkpoints
)

fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True, sharey=True)


def draw(frame_index):
    fraction = checkpoints[frame_index]
    for ax, key in zip(axes.flat, panels):
        ax.clear()
        ax.hist(window_at(*key, fraction), bins=bins, range=(lo, hi), color=MOMENT_COLOR[key[1]])
        for edge in (-THRESHOLD, THRESHOLD):
            ax.axvline(edge, color="k", lw=0.6, ls=":")
        ax.set_xlim(lo, hi)
        ax.set_ylim(0, ymax * 1.05)
        ax.set_title(f"{key[0]} (n={MICE_BY_COHORT[key[0]]}) - {key[1]}", fontsize=9)
    for ax in axes[-1]:
        ax.set_xlabel("d'")
    fig.suptitle(f"Sliding-window d' distribution (size-10 pairs) - session progress {fraction:.2f}")


result = HTML(animation.FuncAnimation(fig, draw, frames=len(checkpoints), interval=300).to_jshtml())
plt.close(fig)
result

In [ ]:
AREA_LABELS = {"V1": "V1", "mHV": "medial (mHV)", "aHV": "anterior (aHV)"}
AREA_CAP = 4000


def session_area_dynamics(session):
    joiner = Joiner(
        db,
        session.recording_id,
        experiment=session.experiment,
        behavior_key=session.behavior_key,
    )
    frames = eligible_trial_frames(joiner.frames)
    pairs = ordinal_trial_pairs(frames)
    windows = paired_trial_windows(pairs, size=10, stride=1)
    progress = windows["progress"].to_numpy()
    dynamics = {}
    for area in AREA_LABELS:
        pool = joiner.neurons[joiner.neurons.area_group == area]
        if len(pool) > AREA_CAP:
            pool = pool.sample(n=AREA_CAP, random_state=0)
        dprime = windowed_svd_dprime(
            joiner.U, joiner.V, frames, pairs, windows, neuron_ids=pool.neuron_id.to_numpy()
        )
        dynamics[area] = (progress, dprime)
    return dynamics


area_records = []
area_distributions = {}
for session in selection.itertuples(index=False):
    for area, (progress, dprime) in session_area_dynamics(session).items():
        for window_index, fraction in enumerate(progress):
            values = dprime[window_index]
            area_records.append({
                "cohort": session.cohort,
                "moment": session.moment,
                "mouse": session.mouse,
                "area": area,
                "progress": float(fraction),
                "mean_abs_dprime": float(np.abs(values).mean()),
                "frac_selective": float((np.abs(values) >= THRESHOLD).mean()),
            })
        area_distributions[(session.cohort, session.mouse, session.moment, area)] = (progress, dprime)

area_history = pd.DataFrame(area_records)
area_history

In [ ]:
AREAS3 = ["V1", "mHV", "aHV"]

fig, axes = plt.subplots(len(AREAS3), len(COHORTS), figsize=(13, 8), sharex=True, sharey=True)
for row_index, area in enumerate(AREAS3):
    for col_index, cohort in enumerate(COHORTS):
        ax = axes[row_index, col_index]
        for moment, color in MOMENT_COLOR.items():
            track = area_history[(area_history.area == area) & (area_history.cohort == cohort) & (area_history.moment == moment)]
            if len(track):
                plot_mouse_average(ax, track, "mean_abs_dprime", color=color, label=moment)
        ax.grid(alpha=0.2)
        if row_index == 0:
            ax.set_title(f"{cohort} (n={MICE_BY_COHORT[cohort]})")
        if col_index == 0:
            ax.set_ylabel(f"{AREA_LABELS[area]}\nmean |d'|")
        if row_index == len(AREAS3) - 1:
            ax.set_xlabel("within-session progress")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Within-session d' by cortical area (size-10 sliding): before vs after - all available matched mice")
plt.show()

In [ ]:
from matplotlib import animation
from IPython.display import HTML

col_keys = [(cohort, moment) for cohort in COHORTS for moment in ("before", "after")]
checkpoints = np.linspace(0.0, 1.0, 24)
area_values = np.concatenate([dprime.ravel() for dprime in (d for _, d in area_distributions.values())])
lo, hi = np.percentile(area_values, [1, 99])
bins = np.linspace(lo, hi, 46)


def area_window_at(cohort, moment, area, fraction):
    values = []
    for (current_cohort, _, current_moment, current_area), (progress, dprime) in area_distributions.items():
        if (current_cohort, current_moment, current_area) == (cohort, moment, area):
            values.append(dprime[int(np.argmin(np.abs(progress - fraction)))])
    return np.concatenate(values)


ymax = max(
    np.histogram(area_window_at(cohort, moment, area, f), bins=bins, range=(lo, hi), density=True)[0].max()
    for area in AREAS3
    for cohort, moment in col_keys
    for f in checkpoints
)

fig, axes = plt.subplots(len(AREAS3), len(col_keys), figsize=(15, 7.5), sharex=True, sharey=True)


def draw(frame_index):
    fraction = checkpoints[frame_index]
    for row_index, area in enumerate(AREAS3):
        for col_index, (cohort, moment) in enumerate(col_keys):
            ax = axes[row_index, col_index]
            ax.clear()
            ax.hist(
                area_window_at(cohort, moment, area, fraction),
                bins=bins, range=(lo, hi), density=True, color=MOMENT_COLOR[moment],
            )
            for edge in (-THRESHOLD, THRESHOLD):
                ax.axvline(edge, color="k", lw=0.5, ls=":")
            ax.set_xlim(lo, hi)
            ax.set_ylim(0, ymax * 1.05)
            if row_index == 0:
                ax.set_title(f"{cohort} (n={MICE_BY_COHORT[cohort]})\n{moment}", fontsize=8)
            if col_index == 0:
                ax.set_ylabel(AREA_LABELS[area], fontsize=9)
    fig.suptitle(f"d' distribution by area x cohort x moment (size-10) - progress {fraction:.2f}")


result = HTML(animation.FuncAnimation(fig, draw, frames=len(checkpoints), interval=300).to_jshtml())
plt.close(fig)
result

In [ ]:
STAT_WINDOW = 20
DIST_STATS = ["mean", "std_dev", "skewness", "kurtosis", "mean_absolute", "median", "pct_selective"]


def area_stat_rows(session):
    joiner = Joiner(
        db,
        session.recording_id,
        experiment=session.experiment,
        behavior_key=session.behavior_key,
    )
    frames = eligible_trial_frames(joiner.frames)
    pairs = ordinal_trial_pairs(frames)
    windows = paired_trial_windows(pairs, size=STAT_WINDOW, stride=1)
    progress = windows["progress"].to_numpy()
    rows = []
    for area in AREA_LABELS:
        pool = joiner.neurons[joiner.neurons.area_group == area]
        if len(pool) > AREA_CAP:
            pool = pool.sample(n=AREA_CAP, random_state=0)
        dprime = windowed_svd_dprime(
            joiner.U, joiner.V, frames, pairs, windows, neuron_ids=pool.neuron_id.to_numpy()
        )
        for window_index, fraction in enumerate(progress):
            values = pd.Series(dprime[window_index])
            rows.append({
                "cohort": session.cohort,
                "moment": session.moment,
                "mouse": session.mouse,
                "area": area,
                "progress": float(fraction),
                "mean": float(values.mean()),
                "std_dev": float(values.std()),
                "skewness": float(values.skew()),
                "kurtosis": float(values.kurt()),
                "mean_absolute": float(values.abs().mean()),
                "median": float(values.median()),
                "pct_selective": float((values.abs() >= THRESHOLD).mean() * 100.0),
            })
    return rows


area_stats = pd.DataFrame(
    [row for session in selection.itertuples(index=False) for row in area_stat_rows(session)]
)
area_stats

In [ ]:
COHORT_COLOR = {"supervised": "#2C6FBB", "unsupervised": "#D1495B", "grating": "#E8A33D"}
MOMENT_STYLE = {"before": ":", "after": "-"}

fig, axes = plt.subplots(len(AREAS3), len(DIST_STATS), figsize=(22, 8.5), sharex=True)
for row_index, area in enumerate(AREAS3):
    for col_index, stat in enumerate(DIST_STATS):
        ax = axes[row_index, col_index]
        for cohort in COHORTS:
            for moment in ("before", "after"):
                track = area_stats[(area_stats.area == area) & (area_stats.cohort == cohort) & (area_stats.moment == moment)]
                if len(track):
                    plot_mouse_average(
                        ax, track, stat, color=COHORT_COLOR[cohort], linestyle=MOMENT_STYLE[moment],
                        label=f"{cohort} {moment}" if (row_index == 0 and col_index == 0) else None,
                    )
        ax.grid(alpha=0.2)
        if row_index == 0:
            ax.set_title(stat)
        if col_index == 0:
            ax.set_ylabel(AREA_LABELS[area], fontsize=10)
        if row_index == len(AREAS3) - 1:
            ax.set_xlabel("progress")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=6, fontsize=8, bbox_to_anchor=(0.5, 1.0))
fig.suptitle(f"d' distribution statistics across within-session windows (size {STAT_WINDOW}), per cortical area - all available matched mice", y=1.05)
plt.tight_layout()
plt.show()